# LAB 02 SOLUTION: Python Bridge, Pandas Intro & Multi-Source Bronze Ingestion
**Course:** Business Intelligence and Big Data Analysis (`01 31 0367 00`)  
**Target Audience:** Tutors & Instructors (Executed Reference Solution)  
**Duration:** 3 Academic Hours (`135 min` / `2h15m` Hands-On)

---

## 🎯 1. Learning & Technical Recap
This executed reference notebook covers all 16 progressive steps across a `2h15m` lab:
1. Step-by-step commented environment & path check (`Python 3.11+`, `.venv`, `REPO_ROOT`, `BRONZE_RAW`, `BRONZE_STAGED`).
2. Guided bridge covering every Python idiom introduced after Lab 01 (`Path`, `with open`, comprehensions, tuples, `.items()`, `assert`, bitwise `&`/`|`/`~` masks).
3. Hands-on **Pandas** conceptual & practical onboarding (vectorized loop speedup benchmark, `.loc[]`, `.str`, `.isna()`, `.groupby()`).
4. Lossless **CSV + JSON + multi-sheet XLSX** Bronze ingestion with audit lineage + Case 3 `pd.json_normalize(raw, sep="_")`.
5. Immutable **Snappy Parquet** export (`index=False`) + disk footprint benchmark & round-trip verification.


## ⚙️ 2. Environment Check & Dynamic Path Discovery
Before touching plant data, a Data Engineer verifies two things: **(1) the exact Python `.venv` kernel and libraries**, and **(2) portable paths** that work identically on Windows, macOS, and Linux.

> **Quick Fix Guide if Step 1 Raises an Error:**
> - **`Python 3.11+ required` or `Missing Python packages`:** Click **Select Kernel** (top-right of VS Code) → **Python Environments...** → **`.venv`**. If `.venv` doesn't exist, run `setup_windows.ps1` / `./setup_macos.sh` / `./setup_linux.sh` in Terminal.
> - **`Could not find '03_labs_student_starters'`:** Use **File → Open Folder...** in VS Code to open the **repository root folder** (`ife_bi_bigdata_student` or `IFE - BI and Big Data`), not a parent directory like `Desktop` or `Documents`.
> - **`No files or Missing raw data`:** Run `python verify_environment.py` in Terminal to inspect the `lab_02_bronze_ingestion_and_profiling/data/` folder.

### Step 1: Verify Libraries, Explain Discovery Logic & Locate Raw Files
Run the cell below without editing it and read the inline Python comments to understand each check.

In [1]:
# ==============================================================================
# STEP 1: ENVIRONMENT SELF-CHECK & DYNAMIC REPOSITORY DISCOVERY
# ==============================================================================
# Read the comments below carefully — every single line is explained step by step.
# If anything fails, the error message will tell you EXACTLY how to fix it!

import sys  # Built-in module that tells us which Python interpreter is running
from pathlib import Path  # Modern object-oriented way to build Windows/Mac/Linux paths

# ------------------------------------------------------------------------------
# 1A. VERIFY PYTHON VERSION (MUST BE 3.11 OR NEWER)
# ------------------------------------------------------------------------------
# sys.version_info is a tuple like (3, 12, 4). Comparing tuples checks major then minor version.
print(f"Python executable : {sys.executable}")
print(f"Python version    : {sys.version.split()[0]}")

if sys.version_info < (3, 11):
    raise RuntimeError(
        f"Python 3.11+ is required, but Jupyter is using {sys.version.split()[0]} ({sys.executable}).\n"
        "HOW TO FIX IN VS CODE:\n"
        "  1. Click 'Select Kernel' (top-right corner of this notebook).\n"
        "  2. Pick 'Python Environments...' -> '.venv' (Python 3.11+).\n"
        "  3. If '.venv' does not exist yet, open a VS Code Terminal and run:\n"
        "     - Windows PowerShell: .\\setup_windows.ps1\n"
        "     - macOS Terminal:     ./setup_macos.sh\n"
        "     - Linux Terminal:     ./setup_linux.sh"
    )

# ------------------------------------------------------------------------------
# 1B. VERIFY ESSENTIAL DATA LIBRARIES ARE INSTALLED IN THIS KERNEL
# ------------------------------------------------------------------------------
# If you accidentally selected global/system Python instead of '.venv', this block catches it early!
_required_libs = {
    "pandas": "DataFrame tables & CSV/JSON/Excel ingestion",
    "openpyxl": "Reading multi-sheet .xlsx Excel workbooks",
    "pyarrow": "Writing compressed columnar .parquet files",
}
_missing_libs = []
for _pkg_name, _purpose in _required_libs.items():
    try:
        # __import__() dynamically imports a library by its string name
        _mod = __import__(_pkg_name)
        print(f"  [OK] {_pkg_name:<10} v{getattr(_mod, '__version__', 'unknown')} -> {_purpose}")
    except ImportError:
        _missing_libs.append(f"{_pkg_name} ({_purpose})")

if _missing_libs:
    raise ImportError(
        "Missing Python packages in the active Jupyter kernel:\n  - "
        + "\n  - ".join(_missing_libs)
        + f"\nActive interpreter: {sys.executable}\n"
        + "HOW TO FIX:\n"
        + "  1. Most common cause: wrong Jupyter kernel! Switch to '.venv' via top-right 'Select Kernel'.\n"
        + "  2. If '.venv' IS already selected, open Terminal with (.venv) active and run:\n"
        + "     pip install -r requirements.txt"
    )

# Import the remaining libraries now that we know the environment is healthy
import json
import time
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display  # Rich interactive HTML table rendering in Jupyter/VS Code

# ------------------------------------------------------------------------------
# 1C. LOCATE REPOSITORY ROOT AUTOMATICALLY (NO HARDCODED C:\ OR /Users/ PATHS!)
# ------------------------------------------------------------------------------
# Why leading underscores like '_cwd' and '_cand'?
# It is a Python naming convention meaning "temporary loop variable you don't need later".
# Path.cwd().resolve() gets the absolute folder where Jupyter started this notebook.
_cwd = Path.cwd().resolve()
REPO_ROOT = None

# What does `*_cwd.parents` mean?
# `_cwd.parents` holds all parent folders upward ([lab_02_dir, 03_labs, repo_root, ...]).
# Placing `*` before `_cwd.parents` UNPACKS those folders into one flat search list after `_cwd`.
for _cand in [_cwd, *_cwd.parents]:
    # Check 1: Are we already inside the course/student repo folder?
    if (_cand / "03_labs_student_starters").exists():
        REPO_ROOT = _cand
        break  # Stop searching as soon as we find the folder containing '03_labs_student_starters'
    # Check 2: What if VS Code opened the parent folder containing the cloned repo?
    for _sub in ("ife_bi_bigdata_course", "ife_bi_bigdata_student"):
        if (_cand / _sub / "03_labs_student_starters").exists():
            REPO_ROOT = _cand / _sub
            break
    if REPO_ROOT is not None:
        break

# `assert condition, error_message` halts immediately with a helpful message if REPO_ROOT is still None
assert REPO_ROOT is not None, (
    f"Could not find '03_labs_student_starters' starting from {_cwd}.\n"
    "HOW TO FIX IN VS CODE:\n"
    "  Go to File -> Open Folder... and open your repository folder directly "
    "(e.g. 'ife_bi_bigdata_student' inside Courses/), then reopen this notebook."
)

# ------------------------------------------------------------------------------
# 1D. CONFIGURE CASE STUDY & SANDBOX OUTPUT FOLDERS (BRONZE STAGING)
# ------------------------------------------------------------------------------
# All Lab 02 guided cells run on SmartPress Automotive by default.
# (Section 10 below also explores Case 3 nested JSON without overwriting these variables!)
CASE_NAME = "case1_smartpress_automotive"
# CASE_NAME = "case2_dataco_supply_chain"   # Optional reference for project work
# CASE_NAME = "case3_nexis_workforce"       # Optional reference for project work

# Note the '/' operator on Path objects: pathlib automatically handles '/' vs '\' across OSes!
STUDENT_BRONZE_DIR = REPO_ROOT / "03_labs_student_starters" / "lab_02_bronze_ingestion_and_profiling" / "data" / CASE_NAME
REF_CASE_DATA = REPO_ROOT / "05_case_studies_reference_solutions" / CASE_NAME / "data"

# Ternary expression (`A if condition else B`):
# Works seamlessly whether you opened the student repo (data in lab_02/data/) or instructor repo (data in 05_case_studies...)
BRONZE_RAW = STUDENT_BRONZE_DIR if STUDENT_BRONZE_DIR.exists() else (REF_CASE_DATA / "bronze")
SANDBOX_DATA = REPO_ROOT / "03_labs_student_starters" / "sandbox_outputs" / CASE_NAME
BRONZE_STAGED = SANDBOX_DATA / "bronze_staged"

# mkdir(parents=True, exist_ok=True) creates missing parent folders AND never crashes if the folder already exists
BRONZE_STAGED.mkdir(parents=True, exist_ok=True)


def resolve_stage_dir(stage_name: str) -> Path:
    """Helper used in Labs 03-10 to locate upstream Parquet outputs in sandbox_outputs first."""
    cand = SANDBOX_DATA / stage_name
    # `any(...)` checks if at least one *.parquet file already exists inside cand
    if cand.exists() and any(cand.glob("*.parquet")):
        return cand
    ref_cand = REF_CASE_DATA / stage_name
    if ref_cand.exists():
        return ref_cand
    raise FileNotFoundError(
        f"Stage '{stage_name}' not found in {cand}. Run the previous Lab notebook cleanly first."
    )


print("\nRepository Root :", REPO_ROOT)
print("Bronze Raw Dir  :", BRONZE_RAW, "| Exists:", BRONZE_RAW.exists())
print("Bronze Staged   :", BRONZE_STAGED)

assert BRONZE_RAW.exists(), (
    f"Missing raw data directory: {BRONZE_RAW}\n"
    "HOW TO FIX: Run 'python verify_environment.py' in VS Code Terminal to check missing datasets."
)

print("\nDetected Raw Source Files:")
for _p in sorted(BRONZE_RAW.iterdir()):
    if _p.is_file():
        # Formatting notes: `:<32` pads text left-aligned to 32 chars; `,.1f` adds comma thousands & 1 decimal
        print(f"  - {_p.name:<32} ({_p.stat().st_size / 1024:,.1f} KB)")


Python executable : <VENV_PYTHON>
Python version    : 3.13.3
  [OK] pandas     v2.3.3 -> DataFrame tables & CSV/JSON/Excel ingestion
  [OK] openpyxl   v3.1.5 -> Reading multi-sheet .xlsx Excel workbooks
  [OK] pyarrow    v26.0.1 -> Writing compressed columnar .parquet files

Repository Root : <REPO_ROOT>
Bronze Raw Dir  : <REPO_ROOT>/05_case_studies_reference_solutions/case1_smartpress_automotive/data/bronze | Exists: True
Bronze Staged   : <REPO_ROOT>/03_labs_student_starters/sandbox_outputs/case1_smartpress_automotive/bronze_staged

Detected Raw Source Files:
  - machines_and_shifts.xlsx         (6.4 KB)
  - maintenance_logs.json            (63.3 KB)
  - sensor_telemetry.csv             (1,460.9 KB)


## 🌉 3. Python Bridge (Beyond Lab 01): Idioms Used in Data Pipelines
Lab 01 covered variables, lists, dicts, `for` loops, and functions. Real data ingestion introduces **a few compact Python patterns** that you will see in every notebook from Lab 02 to Lab 10. Let's master them one small cell at a time!

### Step 2: Portable Paths (`pathlib.Path`), Context Managers (`with open`) & `json.load()`
Why do data engineers use `with open(...) as f:` instead of older `f = open(...)`?
- A **context manager (`with ... as ...:`)** guarantees Python **automatically closes the file handle** as soon as the indented block exits—even if an exception occurs mid-read.
- Notice also `encoding="utf-8"`: explicitly forcing UTF-8 prevents Windows (`cp1252`/`cp1250`) vs macOS/Linux garbled European accents (`ł`, `é`, `€`).
- Inside the `with` block, `json.load(f)` parses JSON text directly into the **Python `list` of `dict`s** you already practiced in Lab 01!

In [2]:
sample_json_path = BRONZE_RAW / "maintenance_logs.json"
print("File name      :", sample_json_path.name)
print("File extension :", sample_json_path.suffix)
print("Parent folder  :", sample_json_path.parent.name)

with open(sample_json_path, "r", encoding="utf-8") as f:
    preview_tickets = json.load(f)

print(f"Loaded type    : {type(preview_tickets).__name__} with {len(preview_tickets)} records")

last_ticket = preview_tickets[-1]
print("Keys of last ticket:", list(last_ticket.keys()))
print("Last ticket payload:", last_ticket)


File name      : maintenance_logs.json
File extension : .json
Parent folder  : bronze
Loaded type    : list with 286 records
Keys of last ticket: ['ticket_id', 'event_date', 'machine_code', 'downtime_type', 'failure_mode', 'repair_duration_min']
Last ticket payload: {'ticket_id': 'CMMS-20269999', 'event_date': '2099-01-01 03:00:00', 'machine_code': 'PRESS_99', 'downtime_type': 'Unplanned', 'failure_mode': 'Phantom Ticket', 'repair_duration_min': 60}


### Step 3: Comprehensions, Tuples, Dict `.items()` & Structured `f-strings`
Four compact Python shortcuts appear heavily in Steps 7–16:
1. **List comprehension:** `[expr for x in seq if condition]` builds a filtered list in one readable line (replacing `.append()` loops).
2. **Tuple `(a, b, c)`:** An immutable sequence often used for fixed sets of tables or column lists.
3. **Dictionary `.items()`:** `for name, val in my_dict.items():` unpacks **both key and value** together inside a loop.
4. **Formatted `f-strings`:** `f"{val:,}"` adds comma separators (`23,164`) and `f"{name:<20}"` aligns columns neatly.

In [3]:
all_files = sorted(BRONZE_RAW.iterdir())
tabular_names_comp = [p.name for p in all_files if p.suffix in (".csv", ".json", ".xlsx")]
print("Matched source files:", tabular_names_comp)

expected_formats = {
    "CSV": "15-min SCADA telemetry rows",
    "JSON": "CMMS semi-structured maintenance tickets",
    "XLSX": "ERP master data tabs (Machines, Shifts, Variants)",
}
for fmt_key, description in expected_formats.items():
    print(f"  {fmt_key:<5} -> {description}")

stream_files = [p.name for p in all_files if p.suffix in (".csv", ".json")]
assert len(stream_files) == 2 and all(name.endswith((".csv", ".json")) for name in stream_files)
print("Non-Excel stream files:", stream_files)


Matched source files: ['machines_and_shifts.xlsx', 'maintenance_logs.json', 'sensor_telemetry.csv']
  CSV   -> 15-min SCADA telemetry rows
  JSON  -> CMMS semi-structured maintenance tickets
  XLSX  -> ERP master data tabs (Machines, Shifts, Variants)
Non-Excel stream files: ['maintenance_logs.json', 'sensor_telemetry.csv']


## 🐼 4. Introduction to Pandas: Why Engineers Use It Across the Whole Course
We will spend Labs 02–08 working heavily in **`pandas`** before pushing Gold Star Schemas to **DuckDB** SQL. Why not just use Excel or pure Python `for` loops?

| Capability | Excel / Spreadsheets | Pure Python (`list[dict]`) | **`pandas` (`DataFrame`)** |
| :--- | :--- | :--- | :--- |
| **Volume Limit** | Hard `1,048,576` row ceiling; freezes > 200k rows | Unlimited RAM, but slow nested loops | Handles millions of rows cleanly in C/NumPy buffers |
| **Execution Model** | Cell-by-cell recalculation | One item per Python loop step | **Vectorized**: transforms entire typed columns at once (10×–100× faster) |
| **Schema & Nulls** | Dates silently flip US/EU; blanks confuse formulas | Manual `try/float()` per dict | Explicit `.dtypes` (`float64`, `datetime64`), native `NaN` propagation |
| **Data Engineering Suite** | No SQL joins, regex, or Parquet | Must hand-code joins & file parsers | Built-in `.read_csv/json/excel/parquet`, `.str`, `.dt`, `.groupby`, `.merge` |

### Step 4: Live Benchmark — Pure Python `for`-Loop vs Pandas Vectorization
Pandas provides two core objects: **`pd.Series`** (a single typed column with an index) and **`pd.DataFrame`** (a 2D table built from aligned `Series`). We will compare pure Python loops against vectorized Pandas on **100,000 simulated press cycles** using three initial variables:
- `n_rows`: Number of simulated 15-minute SCADA production buckets (`100,000`).
- `produced_list`: Total units stamped in each bucket (`200`–`249` units).
- `scrap_list`: Defective/scrapped units in each bucket (`2`–`8` units).

**Part 4A — Pure Python `zip()` & `for`-Loop:** First, see what `zip(produced_list, scrap_list)` pairs together and time the row-by-row loop.

In [4]:
n_rows = 100_000
produced_list = [200 + (i % 50) for i in range(n_rows)]
scrap_list = [2 + (i % 7) for i in range(n_rows)]

zipped_preview = list(zip(produced_list[:5], scrap_list[:5]))
print("First 5 zipped (produced, scrap) pairs:", zipped_preview)

t0 = time.perf_counter()
scrap_pct_py = []
for prod, scr in zip(produced_list, scrap_list):
    scrap_pct_py.append((scr / prod) * 100.0)
py_ms = (time.perf_counter() - t0) * 1000
print(f"Pure Python loop (100k rows): {py_ms:6.2f} ms")


First 5 zipped (produced, scrap) pairs: [(200, 2), (201, 3), (202, 4), (203, 5), (204, 6)]
Pure Python loop (100k rows):   6.79 ms


**Part 4B — Inspecting a Pandas `DataFrame` & Vectorized Benchmark:** Now load the exact same lists into a `pd.DataFrame`, inspect its shape, column types, and first 5 rows (`.head()`), and measure the vectorized speed!

In [5]:
df_bench = pd.DataFrame({"produced": produced_list, "scrap": scrap_list})
t1 = time.perf_counter()
df_bench["scrap_pct"] = (df_bench["scrap"] / df_bench["produced"]) * 100.0
pd_ms = (time.perf_counter() - t1) * 1000

print(f"Pandas vectorized (100k rows): {pd_ms:6.2f} ms  -> Speedup: {py_ms / max(pd_ms, 0.01):.1f}x faster!")
print("\nHow to inspect a DataFrame:")
print("  Shape (.shape)       ->", df_bench.shape, "(rows, columns)")
print("  Columns (.columns)   ->", list(df_bench.columns))
print("  Data types (.dtypes) ->", df_bench.dtypes.to_dict())
print("  Series vs DataFrame  ->", type(df_bench).__name__, "vs", type(df_bench["scrap_pct"]).__name__)
print("\nFirst 5 rows via display(df_bench.head(5)):")
display(df_bench.head(5))


Pandas vectorized (100k rows):   0.88 ms  -> Speedup: 7.8x faster!

How to inspect a DataFrame:
  Shape (.shape)       -> (100000, 3) (rows, columns)
  Columns (.columns)   -> ['produced', 'scrap', 'scrap_pct']
  Data types (.dtypes) -> {'produced': dtype('int64'), 'scrap': dtype('int64'), 'scrap_pct': dtype('float64')}
  Series vs DataFrame  -> DataFrame vs Series

First 5 rows via display(df_bench.head(5)):
   produced  scrap  scrap_pct
0       200      2   1.000000
1       201      3   1.492537
2       202      4   1.980198
3       203      5   2.463054
4       204      6   2.941176


### Step 5: Defensive `assert` / `is not None` Guards & Vectorized Masks (`&`, `|`, `~`)
**Industrial Context:** On the SmartPress shopfloor, SCADA telemetry sensors occasionally log physically impossible shaft torque outliers (such as `-999.0 Nm` from a disconnected sensor or `4850.0 Nm` from a power spike) or events from unregistered/ghost machines (`PRESS_99`). Before computing plant KPIs, we use **vectorized boolean masks** on `pd.Series` to flag and filter out these faulty rows without writing slow loops.

Two critical safety rules protect industrial notebooks from silent bugs:
- **`if df is not None:` and `assert` guards:** Always verify an object was created before calling `.shape` or `.to_parquet()`—otherwise Python crashes with a confusing `NoneType` error several cells later.
- **Python `and`/`or`/`not` vs Pandas `&`/`|`/`~`:**
  - Use **`and`, `or`, `not`** for **single** True/False scalar checks (`if x > 0 and y < 10:`).
  - Use **bitwise `&`, `|`, `~` WITH PARENTHESES** when combining **whole Pandas columns (boolean masks)**: `(s > 200) | (s < 0)`.
  - *Why parentheses?* Without `( ... )`, Python binds `|` tighter than `<`/`>` and fails with `TypeError`!

In [6]:
pressure_bar = 145.0
curtain_ok = True
scalar_safe = (pressure_bar <= 180.0) and curtain_ok
print("Scalar check passed:", scalar_safe)

sample_torques = pd.Series([42.5, -999.0, 58.1, 4850.0, 61.3])
valid_machines = {"PRESS_01", "PRESS_02"}
sample_ids = pd.Series(["PRESS_01", "PRESS_99", "PRESS_02", "PRESS_01", "PRESS_02"])

bad_torque_mask = (sample_torques < 0.0) | (sample_torques > 200.0)
orphan_machine_mask = ~sample_ids.isin(valid_machines)
any_anomaly_mask = bad_torque_mask | orphan_machine_mask

assert int(any_anomaly_mask.sum()) == 2
print("Combined anomaly flags:", any_anomaly_mask.tolist())
display(pd.DataFrame({"id": sample_ids, "torque": sample_torques})[any_anomaly_mask])


Scalar check passed: True
Combined anomaly flags: [False, True, False, True, False]
         id  torque
1  PRESS_99  -999.0
3  PRESS_01  4850.0


### Step 6: Pandas Core Toolkit Sandbox (`.loc[]`, `.isna()`, `.str`, `.value_counts()`, `.groupby()`)
Before touching the 23,164-row telemetry file, practice the five workhorse Pandas operations you will rely on across Labs 02–05 on a tiny 6-row shopfloor sample:
- **`.loc[row_filter, [cols]]`** (*label-based location*, vs `.iloc` for integer positions): Filter rows cleanly without triggering chained-assignment bugs.
- **`.isna().sum()`:** Why `.sum()` to *count* nulls? `.isna()` returns `True`/`False` booleans, and in Python/Pandas `True == 1` and `False == 0`—so `.sum()` literally adds up the `1`s to count missing values (`NaN` is *not* `0`!).
- **`.str.strip().str.upper()` & `.assign()`:** Preview normalized strings *in memory* without altering `mini_plant` (`.assign(col=...)` returns a new copy with an added column).
- **`df['col']`, `.value_counts(dropna=False)` & `.groupby(...)[col].sum()`:** Square brackets `[...]` after a `DataFrame` (or `GroupBy`) select a single `Series` column (or filter rows when passed a boolean `Series`).

In [7]:
mini_plant = pd.DataFrame({
    "machine_id": ["Press_01 ", "P-02", "PRESS_01", "press_03", "PRESS_99", "PRESS_02"],
    "torque_Nm": [54.2, -999.0, 56.8, 49.0, 4850.0, 61.1],
    "process_temp_K": [310.4, 312.0, None, 309.5, 315.2, 311.8],
    "produced_units": [220, 210, 235, 195, 10, 205],
})

print("1. Missing values per column:")
print(mini_plant.isna().sum().to_string())

print("\n2. Preview normalized machine strings (.str accessor):")
clean_names = mini_plant["machine_id"].str.strip().str.upper()
print(clean_names.value_counts().to_string())

mini_outliers = mini_plant.loc[
    (mini_plant["torque_Nm"] < 0) | (mini_plant["torque_Nm"] > 200),
    ["machine_id", "torque_Nm"],
]
mini_grouped = mini_plant.assign(clean_id=clean_names).groupby("clean_id")["produced_units"].sum()

assert len(mini_outliers) == 2
print("\nDetected outliers via .loc[]:")
display(mini_outliers)
print("\nProduced units per normalized machine:")
display(mini_grouped.to_frame("produced_units"))


1. Missing values per column:
machine_id        0
torque_Nm         0
process_temp_K    1
produced_units    0

2. Preview normalized machine strings (.str accessor):
machine_id
PRESS_01    2
P-02        1
PRESS_03    1
PRESS_99    1
PRESS_02    1

Detected outliers via .loc[]:
  machine_id  torque_Nm
1       P-02     -999.0
4   PRESS_99     4850.0

Produced units per normalized machine:
          produced_units
clean_id                
P-02                 210
PRESS_01             455
PRESS_02             205
PRESS_03             195
PRESS_99              10


## 📥 5. Part 1: Ingest SCADA CSV Telemetry & Inject Audit Metadata
In **Bronze**, every ingested table must remain a **lossless mirror** of raw inputs plus two standardized lineage columns:
- `_ingestion_timestamp_utc`: ISO-8601 UTC string from `datetime.now(timezone.utc).isoformat()` recording *when the pipeline captured the snapshot* (distinct from the sensor `timestamp`!).
- `_source_file`: Exact filename (`"sensor_telemetry.csv"`) for downstream audit traceability.

### Step 7: Load `sensor_telemetry.csv` and Inspect Schema & Memory
Complete the `# TODO` lines to read `BRONZE_RAW / csv_name` into `df_telemetry` and attach both audit columns.

> **Why does `.info()` show `Dtype = object` for both `timestamp` and `_ingestion_timestamp_utc`?**  
> In Pandas 2.x, text (`str`) columns are stored as `object` arrays. Calling `.isoformat()` converts `datetime.now(timezone.utc)` into an ISO-8601 **string** (`"2026-...+00:00"`), so *both* columns are currently text strings (`type(val) == str`). You can verify an `object` column's true Python type via `type(df[col].iloc[0])`. We intentionally defer `pd.to_datetime()` conversion to `datetime64` until **Lab 03 (Silver)** so Bronze remains a lossless mirror of raw text.

In [8]:
csv_name = "sensor_telemetry.csv"
now_utc = datetime.now(timezone.utc).isoformat()

df_telemetry = pd.read_csv(BRONZE_RAW / csv_name)
df_telemetry["_ingestion_timestamp_utc"] = now_utc
df_telemetry["_source_file"] = csv_name

assert {"_ingestion_timestamp_utc", "_source_file"}.issubset(df_telemetry.columns)
print(f"Loaded {csv_name} -> {df_telemetry.shape[0]:,} rows x {df_telemetry.shape[1]} columns")
print("\nSchema & RAM usage via .info(memory_usage='deep'):")
df_telemetry.info(memory_usage="deep")
print("\nFirst 3 rows:")
display(df_telemetry.head(3))


Loaded sensor_telemetry.csv -> 23,164 rows x 12 columns

Schema & RAM usage via .info(memory_usage='deep'):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23164 entries, 0 to 23163
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   timestamp                 23164 non-null  object 
 1   machine_id                23164 non-null  object 
 2   product_variant           23164 non-null  object 
 3   air_temp_K                23164 non-null  float64
 4   process_temp_K            23044 non-null  float64
 5   rotational_speed_rpm      23164 non-null  int64  
 6   torque_Nm                 23164 non-null  float64
 7   tool_wear_min             23164 non-null  int64  
 8   produced_units            23164 non-null  int64  
 9   scrap_units               23164 non-null  int64  
 10  _ingestion_timestamp_utc  23164 non-null  object 
 11  _source_file              23164 non-null  object 
dtypes: float

### Step 8: First Inspection of Ingested Telemetry (Time Window, Categories & Distributions)
Notice in `.info()` above that `timestamp` arrived as `object`/`str`—CSV stores pure text, not native timestamps! Inspect min/max timestamp text, steel variant counts, and numeric summaries before moving to JSON.

In [9]:
ts_min = df_telemetry["timestamp"].min()
ts_max = df_telemetry["timestamp"].max()
variant_counts = df_telemetry["product_variant"].value_counts(dropna=False)

print(f"Raw timestamp range: {ts_min}  -->  {ts_max}  (notice 2099-* glitch at the top!)")
print("\nSteel product_variant distribution:")
print(variant_counts.to_string())
print("\nSummary statistics (.describe()):")
print(df_telemetry[["air_temp_K", "process_temp_K", "torque_Nm", "tool_wear_min", "produced_units", "scrap_units"]].describe().round(2).to_string())


Raw timestamp range: 2026-10-01 06:00:00  -->  2099-01-01 12:00:00  (notice 2099-* glitch at the top!)

Steel product_variant distribution:
product_variant
L    8975
M    8507
H    5682

Summary statistics (.describe()):
       air_temp_K  process_temp_K  torque_Nm  tool_wear_min  produced_units  scrap_units
count    23164.00        23044.00   23164.00       23164.00        23164.00     23164.00
mean       298.15          307.81      47.37         104.89          195.68         4.69
std          1.27            2.26     145.05          55.08           64.10         3.89
min        293.96          299.28    -999.00          14.00          -50.00         0.00
25%        297.21          306.65      42.66          57.00          179.00         3.00
50%        298.15          307.96      46.11         103.00          211.00         4.00
75%        299.10          309.19      49.58         151.00          240.00         5.00
max        302.35          318.85    4850.00         249.00        

## 📋 6. Part 2: Ingest Semi-Structured JSON Maintenance Logs (`maintenance_logs.json`)
CMMS systems export event tickets as JSON arrays rather than flat CSV grids.

### Step 9: Parse Flat JSON with Context Manager (`with open`) -> `df_maintenance`
Apply the pattern from Step 2: open `BRONZE_RAW / json_name`, parse `raw_records = json.load(f)`, convert to `pd.DataFrame(raw_records)`, and attach `_ingestion_timestamp_utc` and `_source_file`.

In [10]:
json_name = "maintenance_logs.json"
with open(BRONZE_RAW / json_name, "r", encoding="utf-8") as f:
    raw_records = json.load(f)

df_maintenance = pd.DataFrame(raw_records)
df_maintenance["_ingestion_timestamp_utc"] = now_utc
df_maintenance["_source_file"] = json_name

assert {"_ingestion_timestamp_utc", "_source_file"}.issubset(df_maintenance.columns)
print(f"Loaded {json_name} -> {df_maintenance.shape[0]:,} tickets x {df_maintenance.shape[1]} columns")
print("Columns:", list(df_maintenance.columns))
print("\nFirst 5 maintenance tickets:")
display(df_maintenance.head(5))


Loaded maintenance_logs.json -> 286 tickets x 8 columns
Columns: ['ticket_id', 'event_date', 'machine_code', 'downtime_type', 'failure_mode', 'repair_duration_min', '_ingestion_timestamp_utc', '_source_file']

First 5 maintenance tickets:
       ticket_id  ...           _source_file
0  CMMS-20260001  ...  maintenance_logs.json
1  CMMS-20260002  ...  maintenance_logs.json
2  CMMS-20260003  ...  maintenance_logs.json
3  CMMS-20260004  ...  maintenance_logs.json
4  CMMS-20260005  ...  maintenance_logs.json

[5 rows x 8 columns]


### Step 10 (Case Study #3 Skill): Flattening Nested JSON Objects (`pd.json_normalize`)
The core workflow above uses **Case 1 (`SmartPress`)**, where `maintenance_logs.json` is already a flat list of dicts. In **Case 3 (`Nexis Workforce`)**, `workforce_events.json` stores attributes inside a nested **`details`** dictionary (including sensitive PII test fields like `details.medical_note`).

> **Case 2 & Case 3 Quick Source Mapping Note:**
> - **Case 2 (`DataCo Hydraulics`):** `order_items_trans.csv` (ERP order lines), `shipment_events.json` (WMS/3PL milestone events carrying `warehouse_hub`, `shipping_mode`, `carrier_code`, `freight_cost_eur`, `sla_penalty_eur`), and `master_catalog.xlsx` (`Products`, `Customers`, `Warehouses`, `Carrier_SLA_Policies`).
> - **Case 3 (`Nexis Workforce`):** `shift_production_performance.csv` (MES counters including anonymous `spot_fill` temporary workers on `268/720` shifts), `workforce_events.json` (badged events for the `165` roster workers with nested `details`), and `workforce_roster_and_skills.xlsx` (`Employees`, `Skill_Matrix`, `Incentive_Policies`, `Shift_Staffing_Plan`).

Compare naive `pd.DataFrame(raw)` vs `pd.json_normalize(raw, sep="_")` on **Case 3 `workforce_events.json`** (without touching your Case 1 variables):

In [11]:
c3_student = REPO_ROOT / "03_labs_student_starters" / "lab_02_bronze_ingestion_and_profiling" / "data" / "case3_nexis_workforce" / "workforce_events.json"
c3_ref = REPO_ROOT / "05_case_studies_reference_solutions" / "case3_nexis_workforce" / "data" / "bronze" / "workforce_events.json"
c3_json_path = c3_student if c3_student.exists() else c3_ref

with open(c3_json_path, "r", encoding="utf-8") as f:
    c3_raw_events = json.load(f)

df_c3_naive = pd.DataFrame(c3_raw_events[:5])
print("Naive pd.DataFrame keeps nested dict column 'details':")
print("  Columns:", list(df_c3_naive.columns))
print("  Row 0 details:", df_c3_naive["details"].iloc[0])

df_c3_flat = pd.json_normalize(c3_raw_events, sep="_")
detail_cols = sorted(c for c in df_c3_flat.columns if c.startswith("details_"))
print(f"\npd.json_normalize(..., sep='_') -> {df_c3_flat.shape[0]:,} rows x {df_c3_flat.shape[1]} columns")
print(f"Flattened 'details_*' columns ({len(detail_cols)}):", detail_cols)
print("Note: 'details_medical_note' is sensitive PII that MUST be dropped in Lab 03/04 Silver!")


Naive pd.DataFrame keeps nested dict column 'details':
  Columns: ['event_id', 'timestamp', 'employee_id', 'event_type', 'assembly_line_id', 'shift_code', 'badge_reader_id', 'details']
  Row 0 details: {'scheduled_start': '06:00:00', 'is_overtime_early_start': True}

pd.json_normalize(..., sep='_') -> 25,585 rows x 26 columns
Flattened 'details_*' columns (19): ['details_approved_overtime_hours', 'details_clock_in_after_out', 'details_corrupt_clock', 'details_defect_count', 'details_duplicate_scan_flag', 'details_individual_scrap_rate_pct', 'details_is_overtime_early_start', 'details_medical_note', 'details_overtime_reason', 'details_prior_7d_ot_at_event', 'details_qualifies_bonus_threshold', 'details_reason', 'details_reported_hours_prior', 'details_role_assigned', 'details_rolling_7d_ot_hours', 'details_scheduled_start', 'details_station_id', 'details_total_shift_hours', 'details_units_inspected']
Note: 'details_medical_note' is sensitive PII that MUST be dropped in Lab 03/04 Silver!

## 📊 7. Part 3: Ingest Multi-Sheet ERP Excel Master Data
Why does ERP Master Data arrive in multiple Excel tabs inside `machines_and_shifts.xlsx` instead of one sheet?
Because **`Machines`**, **`Shifts`**, and **`Product_Variants`** are distinct business entities with independent primary keys (which will become **Kimball Dimension tables** in Lab 06). Notice how looping over a **tuple** `(df_machines, df_shifts, df_product_variants)` lets us attach audit columns in one clean loop without copy-pasting!

### Step 11: Inspect Excel Sheets & Load All Three Reference Tables

In [12]:
excel_name = "machines_and_shifts.xlsx"
excel_path = BRONZE_RAW / excel_name
sheet_names = pd.ExcelFile(excel_path).sheet_names
print("Detected Excel Sheets:", sheet_names)

df_machines = pd.read_excel(excel_path, sheet_name="Machines")
df_shifts = pd.read_excel(excel_path, sheet_name="Shifts")
df_product_variants = pd.read_excel(excel_path, sheet_name="Product_Variants")

for df_ref in (df_machines, df_shifts, df_product_variants):
    df_ref["_ingestion_timestamp_utc"] = now_utc
    df_ref["_source_file"] = excel_name

assert len(df_machines) == 4 and len(df_shifts) == 3 and len(df_product_variants) == 3
print(f"\nMachines ({df_machines.shape[0]} rows):")
display(df_machines)
print(f"\nShifts ({df_shifts.shape[0]} rows):")
display(df_shifts)
print(f"\nProduct_Variants ({df_product_variants.shape[0]} rows):")
display(df_product_variants)


Detected Excel Sheets: ['Machines', 'Shifts', 'Product_Variants']

Machines (4 rows):
  machine_id  ...              _source_file
0   PRESS_01  ...  machines_and_shifts.xlsx
1   PRESS_02  ...  machines_and_shifts.xlsx
2   PRESS_03  ...  machines_and_shifts.xlsx
3   PRESS_04  ...  machines_and_shifts.xlsx

[4 rows x 7 columns]

Shifts (3 rows):
  shift_id  ...              _source_file
0       S1  ...  machines_and_shifts.xlsx
1       S2  ...  machines_and_shifts.xlsx
2       S3  ...  machines_and_shifts.xlsx

[3 rows x 7 columns]

Product_Variants (3 rows):
  variant_code  ...              _source_file
0            L  ...  machines_and_shifts.xlsx
1            M  ...  machines_and_shifts.xlsx
2            H  ...  machines_and_shifts.xlsx

[3 rows x 7 columns]


## 💾 8. Part 4: Stage All Tables to Snappy Parquet (`index=False`)
Why convert CSV/JSON/Excel into **Apache Parquet (`.parquet`)** for Bronze staging?
- **Columnar storage:** Analytical queries in Pandas & DuckDB only read the columns needed rather than scanning every byte.
- **Embedded schema:** Preserves `int64`, `float64`, and string types cleanly (no re-guessing types from CSV text!).
- **Snappy compression + `index=False`:** Cuts file size dramatically while **`index=False`** prevents Pandas from writing an unwanted `__index_level_0__` column.

### Step 12: Export All 5 Bronze Tables Using Dict `.items()` Iteration

In [13]:
outputs = {
    "bronze_telemetry.parquet": df_telemetry,
    "bronze_maintenance.parquet": df_maintenance,
    "bronze_machines.parquet": df_machines,
    "bronze_shifts.parquet": df_shifts,
    "bronze_product_variants.parquet": df_product_variants,
}

missing_tables = [fname for fname, df_obj in outputs.items() if df_obj is None]
assert not missing_tables, f"Incomplete upstream tables: {missing_tables}"

print("Verified Staged Parquet Artifacts:")
for fname, df_out in outputs.items():
    out_path = BRONZE_STAGED / fname
    df_out.to_parquet(out_path, index=False, compression="snappy")
    print(f"  [OK] {fname:<34} {df_out.shape[0]:>7,} rows -> {out_path.stat().st_size / 1024:>8.2f} KB")


Verified Staged Parquet Artifacts:
  [OK] bronze_telemetry.parquet            23,164 rows ->   264.87 KB
  [OK] bronze_maintenance.parquet             286 rows ->     9.91 KB
  [OK] bronze_machines.parquet                  4 rows ->     5.34 KB
  [OK] bronze_shifts.parquet                    3 rows ->     4.88 KB
  [OK] bronze_product_variants.parquet          3 rows ->     5.21 KB


### Step 13 (Wrap-Up): Verify Parquet Round-Trip (`No Index Leak`) & Compression Savings
Reload `bronze_telemetry.parquet` to confirm zero phantom index columns (`__index_level_0__`), preserved dtypes, and compare raw CSV/JSON footprint vs staged Parquet size.

> **Final Verification Rule:** Click **Restart Kernel and Run All Cells** before finishing Lab 02 to guarantee top-to-bottom reproducibility! Next up in **Lab 03**: Bronze Data Quality Profiling & Silver Cleaning Part 1.

In [14]:
raw_kb = sum(p.stat().st_size for p in BRONZE_RAW.iterdir() if p.is_file()) / 1024
staged_files = sorted(BRONZE_STAGED.glob("*.parquet"))
staged_kb = sum(p.stat().st_size for p in staged_files) / 1024

df_roundtrip = pd.read_parquet(BRONZE_STAGED / "bronze_telemetry.parquet")
assert "__index_level_0__" not in df_roundtrip.columns
compression_ratio = raw_kb / staged_kb
print(f"Raw folder total    : {raw_kb:,.1f} KB")
print(f"Staged Parquet total: {staged_kb:,.1f} KB  ({len(staged_files)} files)")
print(f"Footprint reduction : {compression_ratio:.2f}x smaller than raw CSV/JSON/XLSX")
print("Round-trip shape    :", df_roundtrip.shape, "| index columns intact (no __index_level_0__ leak).")
print("\n-> Lab 02 Bronze Staging complete! Proceed to Lab 03 for Bronze DQ Profiling & Silver Cleaning.")


Raw folder total    : 1,530.7 KB
Staged Parquet total: 290.2 KB  (5 files)
Footprint reduction : 5.27x smaller than raw CSV/JSON/XLSX
Round-trip shape    : (23164, 12) | index columns intact (no __index_level_0__ leak).

-> Lab 02 Bronze Staging complete! Proceed to Lab 03 for Bronze DQ Profiling & Silver Cleaning.
